# Create OeNB Jubilaeumsfonds Awards (Oesterreichische Nationalbank Anniversary Fund)

Creates awards for the Oesterreichische Nationalbank's research fund, the
Jubilaeumsfonds (Anniversary Fund, est. 1966), from the fund's own public
"Projektabfrage" at
https://www.oenb.at/daten-und-forschung/forschungsfoerderung/jubilaeumsfonds/projektabfrage.html .
The page is a thin client over a public JSON API (`/fomis/suche`, `/fomis/detail/{id}`);
the script reads every project through it: project number, title, PI (name + academic
title), research institute, submission and approval dates, description, short report,
publications, discipline / thematic cluster and keywords.
**906 projects, project numbers 14744-19171, approved 2012-2026** (local run 2026-10-01).

**Coverage:** the fund is still active (calls through 2027), but the online database
only holds the projects approved in recent years. The fund's earlier ~14,000 projects
(1966-~2012, numbers below 14744) are not published grant-by-grant, so citation stubs
with older numbers stay as acknowledgement shells.

**Amount waiver (section 6.7):** OeNB publishes no per-project amounts (not in the
database, not in the "Vergabesitzung" approval PDFs); the fund's rules cap projects at
EUR 50,000-300,000. `amount` and `currency` are NULL.

**Dates:** `start_date` = the approval date (`bewilligtDate`, the award decision);
the database has no project period, so `end_date` is NULL.

**Prerequisites:**
- Run `scripts/local/oenb_jubilaeumsfonds_to_s3.py` first. It uploads
  `s3://openalex-ingest/awards/oenb_jubilaeumsfonds/oenb_jubilaeumsfonds_projects.parquet`
  (section 1.4 shrink guard).

**`funder_award_id` (section 2.1.1):** the Jubilaeumsfonds project number as a bare
integer string (`18740`). Citation-side sample (`openalex_awards_raw`, F4320322325,
priority 0): bare 5-digit numbers dominate (1,156 shells; 768 of them, 319 distinct
numbers, fall in this database's 14744-19171 range), so those shells are upgraded.
Prefixed forms (`OeNB 15174`, `P21004`) do not collapse; `P`-prefixed numbers are
mostly FWF grants mis-attributed to OeNB.

**Funder details (Path A, F4320* Crossref-registered):**
- funder_id: `4320322325`
- display_name: Oesterreichische Nationalbank
- ror_id / doi: from `openalex.funders.funders` (ror 00nkcmy90, doi 10.13039/501100004061)

**Priority:** `529` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.oenb_jubilaeumsfonds_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/oenb_jubilaeumsfonds/oenb_jubilaeumsfonds_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_projects FROM openalex.awards.oenb_jubilaeumsfonds_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.oenb_jubilaeumsfonds_raw;

In [ ]:
%sql
SELECT funder_award_id, title, approved_date, pi_title, pi_given_name, pi_family_name, institute, discipline
FROM openalex.awards.oenb_jubilaeumsfonds_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)
F4320322325 is a Crossref-registered funder, so it MUST resolve to exactly 1
row in `openalex.funders.funders`. If 0 rows, STOP (do not proceed) and flag.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders
WHERE funder_id = 4320322325;

## Step 2: Create OeNB Jubilaeumsfonds Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.oenb_jubilaeumsfonds_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320322325  -- Oesterreichische Nationalbank
),

src AS (
    SELECT *,
        named_struct(
            'given_name', NULLIF(TRIM(pi_given_name), ''),
            'family_name', NULLIF(TRIM(pi_family_name), ''),
            'orcid', CAST(NULL AS STRING),
            'role_start', CAST(NULL AS DATE),
            'affiliation', named_struct(
                'name', NULLIF(TRIM(institute), ''),
                'country', CAST(NULL AS STRING),
                'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
            )
        ) AS pi
    FROM openalex.awards.oenb_jubilaeumsfonds_raw
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000 as id,
    NULLIF(TRIM(g.title), '') as display_name,
    COALESCE(NULLIF(TRIM(g.description), ''), NULLIF(TRIM(g.short_report), '')) as description,
    f.funder_id,
    TRIM(g.funder_award_id) as funder_award_id,
    CAST(NULL AS DOUBLE) as amount,
    CAST(NULL AS STRING) as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    'research' as funding_type,
    CONCAT('Jubiläumsfonds', CASE WHEN g.discipline IS NOT NULL THEN CONCAT(' - ', g.discipline) ELSE '' END) as funder_scheme,
    'oenb_jubilaeumsfonds_projektabfrage' as provenance,
    TRY_TO_DATE(g.approved_date, 'yyyy-MM-dd') as start_date,
    CAST(NULL AS DATE) as end_date,
    YEAR(TRY_TO_DATE(g.approved_date, 'yyyy-MM-dd')) as start_year,
    CAST(NULL AS INT) as end_year,
    CASE WHEN g.pi.family_name IS NOT NULL THEN g.pi END as lead_investigator,
    CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
         affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) as co_lead_investigator,
    CASE WHEN g.pi.family_name IS NOT NULL THEN array(g.pi) END as investigators,
    g.landing_page_url as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM src g
CROSS JOIN src_funder f
WHERE g.funder_award_id IS NOT NULL AND TRIM(g.funder_award_id) != '';

In [ ]:
%sql
-- Shape check (section 2.1.1): every funder_award_id is a bare project number; 0 rows expected.
SELECT funder_award_id FROM openalex.awards.oenb_jubilaeumsfonds_awards
WHERE NOT funder_award_id RLIKE '^[0-9]{4,5}$';

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'oenb_jubilaeumsfonds_projektabfrage' AND priority = 529;

-- Insert into openalex_awards_raw with priority.
-- Priority 529: direct-from-funder ingest of the OeNB Jubilaeumsfonds project database.
-- Higher wins under the 2026-06-20 DESC dedup (oxjob #500), so 529 outranks the
-- acknowledgement shells (priority 0) that share these award ids.
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    529 as priority
FROM openalex.awards.oenb_jubilaeumsfonds_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_awards, COUNT(DISTINCT id) as distinct_ids
FROM openalex.awards.oenb_jubilaeumsfonds_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, start_date,
       lead_investigator.given_name, lead_investigator.family_name,
       lead_investigator.affiliation.name
FROM openalex.awards.oenb_jubilaeumsfonds_awards LIMIT 20;

In [ ]:
%sql
SELECT funder_scheme, COUNT(*) as cnt
FROM openalex.awards.oenb_jubilaeumsfonds_awards
GROUP BY funder_scheme ORDER BY cnt DESC;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt
FROM openalex.awards.oenb_jubilaeumsfonds_awards
WHERE start_year IS NOT NULL GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- Section 6.4a frequency check: PI top-20 must be a real long-tail.
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.oenb_jubilaeumsfonds_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Section 6.4a: titles should be distinct.
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.oenb_jubilaeumsfonds_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Section 6.3 / 6.7 coverage (amount waived: not published by OeNB).
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    COUNT(start_date) as has_start_date,
    COUNT(lead_investigator.family_name) as has_pi_name,
    COUNT(lead_investigator.affiliation.name) as has_institution
FROM openalex.awards.oenb_jubilaeumsfonds_awards;

In [ ]:
%sql
-- Acknowledgement shells (priority 0) that this ingest upgrades.
SELECT c.provenance, COUNT(*) as shell_rows, COUNT(t.id) as upgraded
FROM openalex.awards.openalex_awards_raw c
LEFT JOIN openalex.awards.oenb_jubilaeumsfonds_awards t ON t.id = c.id
WHERE c.funder_id = 4320322325 AND c.priority = 0
GROUP BY c.provenance ORDER BY shell_rows DESC;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'oenb_jubilaeumsfonds_projektabfrage'
GROUP BY provenance, priority;